# ALL-IDB1 CNN-GNN 100-Epoch Evaluation

This notebook records the 100-epoch CNN-GNN attempt on the same prepared ALL-IDB1 dataset used by the other notebooks. The run uses the local `data/processed/all_idb1` files only; no additional image data or pretrained weights are used.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))

from leukemia_osl.preprocess import build_manifest, summarize_records, validate_prepared_all_idb1

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "all_idb1"
RESULT_DIR = PROJECT_ROOT / "results" / "hybrid_cnn_gnn_vgg_enhanced_content_100ep"
CHECKPOINT_DIR = PROJECT_ROOT / "results" / "checkpoints" / "hybrid_cnn_gnn_vgg_enhanced_content_100ep"

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Result dir:", RESULT_DIR)


Project: /Users/panshulaj/Documents/AIML proj
Dataset: /Users/panshulaj/Documents/AIML proj/data/processed/all_idb1
Result dir: /Users/panshulaj/Documents/AIML proj/results/hybrid_cnn_gnn_vgg_enhanced_content_100ep


In [2]:
split_counts = validate_prepared_all_idb1(DATA_DIR)
records = build_manifest(DATA_DIR)
image_count = len(records)
class_counts = summarize_records(records)
split_counts, image_count, class_counts

({'train': {'healthy': 41, 'leukemia': 34},
  'val': {'healthy': 8, 'leukemia': 7},
  'test': {'healthy': 10, 'leukemia': 8}},
 108,
 {'healthy': 59, 'leukemia': 49})

## GNN Architecture

- Input: 224x224 RGB microscopy mosaic from the same ALL-IDB1 image files.
- Preprocessing: annotation-color suppression, microscopy white balance, leukocyte/nucleus segmentation mosaic, fold-train-only normalization, and train-time flips/affine/color/autocontrast/sharpness augmentation.
- CNN backbone: VGG11-BN feature extractor, randomly initialized, trainable end-to-end.
- Projection: 1x1 convolution from 512 channels to 192 channels, BatchNorm, GELU.
- Graph nodes: each spatial CNN feature-map position becomes one node.
- Graph edges: 8-neighbor image-grid adjacency with self loops, row-normalized.
- GNN encoder: 4 spatial graph convolution blocks with self/neighbor projections, GELU, dropout 0.20, residual LayerNorm, and a 192 -> 384 -> 192 feed-forward block.
- Readout/head: mean pool plus max pool over graph nodes, LayerNorm, Linear 384 -> 192, GELU, dropout 0.45, Linear 192 -> 2.
- Training: AdamW, lr 0.00025, weight decay 0.0004, class weights, label smoothing 0.02, gradient clipping 1.0, validation-selected threshold, and test-time augmentation.

## Exact GNN Run

The command below is the completed 100-epoch run. It uses VGG11-BN as the CNN feature extractor, graph message passing over the spatial feature-map grid, enhanced preprocessing, and content-grouped cross-validation. This is less constrained than the acquisition-grouped protocol, but still uses the same 108 ALL-IDB1 images.

In [3]:
completed_command = " ".join([
    "env PYTHONPATH=src .venv/bin/python scripts/run_grouped_experiment.py",
    "--model-name hybrid_cnn_gnn",
    "--result-name hybrid_cnn_gnn_vgg_enhanced_content_100ep",
    "--checkpoint-name hybrid_cnn_gnn_vgg_enhanced_content_100ep",
    "--grouping content",
    "--profile gnn_microscopy_enhanced",
    "--backbone vgg11_bn",
    "--image-size 224",
    "--batch-size 16",
    "--epochs 100",
    "--patience 100",
    "--inner-splits 4",
    "--folds 5",
    "--embed-dim 192",
    "--gnn-layers 4",
    "--gnn-dropout 0.20",
    "--graph-neighbors 8",
    "--dropout 0.45",
    "--lr 0.00025",
    "--weight-decay 0.0004",
    "--label-smoothing 0.02",
    "--max-grad-norm 1.0",
    "--lr-patience 8",
    "--threshold-policy val_balanced",
    "--test-time-augmentation",
    "--seed 42",
])
completed_command

'env PYTHONPATH=src .venv/bin/python scripts/run_grouped_experiment.py --model-name hybrid_cnn_gnn --result-name hybrid_cnn_gnn_vgg_enhanced_content_100ep --checkpoint-name hybrid_cnn_gnn_vgg_enhanced_content_100ep --grouping content --profile gnn_microscopy_enhanced --backbone vgg11_bn --image-size 224 --batch-size 16 --epochs 100 --patience 100 --inner-splits 4 --folds 5 --embed-dim 192 --gnn-layers 4 --gnn-dropout 0.20 --graph-neighbors 8 --dropout 0.45 --lr 0.00025 --weight-decay 0.0004 --label-smoothing 0.02 --max-grad-norm 1.0 --lr-patience 8 --threshold-policy val_balanced --test-time-augmentation --seed 42'

## Measured Metrics

In [4]:
metrics = json.loads((RESULT_DIR / "metrics.json").read_text(encoding="utf-8"))
summary = pd.Series({
    "accuracy": metrics["Final/Accuracy"],
    "balanced_accuracy": metrics["Final/Balanced_Accuracy"],
    "f1_macro": metrics["Final/F1_Macro"],
    "mcc": metrics["Final/MCC"],
    "roc_auc_macro": metrics["Final/ROC_AUC_Macro"],
    "test_samples": metrics["Final/Test_Samples"],
    "bootstrap_groups": metrics["Final/Bootstrap_Groups"],
})
summary.to_frame("value")

,value
accuracy,0.935185
balanced_accuracy,0.933760
f1_macro,0.934506
mcc,0.869175
roc_auc_macro,0.993428
test_samples,108.000000
bootstrap_groups,107.000000


In [5]:
fold_metrics = pd.read_csv(RESULT_DIR / "fold_metrics.csv")
fold_metrics[[
    "Fold",
    "Final/Accuracy",
    "Final/Balanced_Accuracy",
    "Training/Best_Epoch",
    "Training/Generalization_Gap",
    "Decision/Threshold",
]]

,Fold,Final/Accuracy,Final/Balanced_Accuracy,Training/Best_Epoch,Training/Generalization_Gap,Decision/Threshold
0,1,1.000000,1.000000,47,0.090909,0.345708
1,2,0.954545,0.950000,58,0.000000,0.500000
2,3,0.904762,0.909091,17,-0.061538,0.500000
3,4,0.863636,0.850000,32,0.090909,0.926390
4,5,0.952381,0.958333,9,-0.046154,0.500000


In [6]:
predictions = pd.read_csv(RESULT_DIR / "predictions.csv")
groups = pd.read_csv(RESULT_DIR / "evaluation_groups.csv")
group_columns = [column for column in groups.columns if column != "image_path"]
group_column = group_columns[0]
prediction_groups = predictions.merge(groups, on="image_path", how="left", validate="one_to_one")
prediction_groups.groupby(group_column)[["is_correct"]].agg(["count", "sum", "mean"])

is_correct         
                                                        count sum mean
content_group                                                         
000621249ebe2cd92f017d64a98dad964c0fef6c5c74253...          1   1  1.0
022ed844df9bcbd1daea53903812d51ee172d2434729073...          1   1  1.0
03d07990bdf70038d919956cbd4ee9dcd347225a0dcf3d1...          1   1  1.0
086570b1c36db1db7dd5153c29eb3fbc3052532c0aeb56c...          1   1  1.0
0910ed28943b97e2db4859cd91c57df681e6d72c3c31f4a...          1   1  1.0
...                                                       ...  ..  ...
f5be0bc211ea1989babce4d914742200581e5fa19d1e7da...          1   1  1.0
f8e17500a244ab5e160fe2237a7a59f93c4d19cd171baea...          1   1  1.0
fbce831a30e93d25df1188709dacafff6bb9f6807ec7f94...          1   1  1.0
fbf6d868a4ebe083677a967362329c645fa66dc6cb2d62f...          1   1  1.0
ff3b9582c03fd06d9aba7a80adea8d4322451a002725f4a...          1   1  1.0

[107 rows x 3 columns]

In [7]:
errors = prediction_groups[~prediction_groups["is_correct"]].copy()
errors[[
    group_column,
    "image_path",
    "true_label",
    "predicted_label",
    "probability_healthy",
    "probability_leukemia",
]].head(40)

,content_group,image_path,true_label,predicted_label,probability_healthy,probability_leukemia
34,9367d850ee64760587d6590ab769c8c4824f8f41ac17e6...,/Users/panshulaj/Documents/AIML proj/data/proc...,leukemia,healthy,0.806130,0.193870
50,a255715173a41c373183e569ad77ea415ed75b66820949...,/Users/panshulaj/Documents/AIML proj/data/proc...,healthy,leukemia,0.016370,0.983630
52,58dce42fa1b553204c713e23b10f9ac113dba76dffc58a...,/Users/panshulaj/Documents/AIML proj/data/proc...,healthy,leukemia,0.023168,0.976832
82,30c68ded18e8ef090a53713aa2d27b5940fdbca70f290e...,/Users/panshulaj/Documents/AIML proj/data/proc...,leukemia,healthy,0.086813,0.913187
83,dda8e2dfe2d83abb7fcdeb4b188e382f05e52010db51a5...,/Users/panshulaj/Documents/AIML proj/data/proc...,leukemia,healthy,0.205327,0.794673
86,5bc8edbc689348a57e958f778e912efaf6c1f0e4af1726...,/Users/panshulaj/Documents/AIML proj/data/proc...,leukemia,healthy,0.928746,0.071254
93,f0b8380ba93f8a98a45d310505764a46ddc548d3afce62...,/Users/panshulaj/Documents/AIML proj/data/proc...,healthy,leukemia,0.082130,0.917870


## Conclusion

The completed 100-epoch CNN-GNN run used the same ALL-IDB1 files and improved substantially over the earlier strict GNN attempt, but its measured final accuracy is still below the requested 98%. This notebook reports the actual saved metrics rather than replacing them with an unsupported number.